In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric = pd.read_csv("metric.csv")
trace = pd.read_csv("trace.csv")
log = pd.read_csv("log.csv")

# Parse timestamps as UTC datetimes (Unix seconds)
for df in (metric, trace, log):
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# 1) metric.csv summary: group by (cmdb_id, kpi_name)
metric_grp = (
    metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        count=('value', 'count'),
        earliest_ts=('timestamp', 'min'),
        latest_ts=('timestamp', 'max'),
        p95_value=('value', lambda s: s.quantile(0.95))
    )
)

# Format timestamps as compact ISO UTC strings
metric_grp['earliest'] = metric_grp['earliest_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
metric_grp['latest'] = metric_grp['latest_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
metric_summary = metric_grp[['cmdb_id', 'kpi_name', 'count', 'earliest', 'latest', 'p95_value']].sort_values(
    ['count', 'cmdb_id', 'kpi_name'], ascending=[False, True, True]
).head(50)

# 2) trace.csv summary: group by (cmdb_id, trace_name)
trace_grp = (
    trace
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(
        count=('value', 'count'),
        earliest_ts=('timestamp', 'min'),
        latest_ts=('timestamp', 'max'),
        p95_value=('value', lambda s: s.quantile(0.95))
    )
)

trace_grp['earliest'] = trace_grp['earliest_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
trace_grp['latest'] = trace_grp['latest_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
trace_summary = trace_grp[['cmdb_id', 'trace_name', 'count', 'earliest', 'latest', 'p95_value']].sort_values(
    ['count', 'cmdb_id', 'trace_name'], ascending=[False, True, True]
).head(50)

# 3) log.csv summary: group by (cmdb_id, log_name)
log_grp = (
    log
    .groupby(['cmdb_id', 'log_name'], as_index=False)
    .agg(
        count=('value', 'count'),
        earliest_ts=('timestamp', 'min'),
        latest_ts=('timestamp', 'max'),
        p95_value=('value', lambda s: s.quantile(0.95))
    )
)

log_grp['earliest'] = log_grp['earliest_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
log_grp['latest'] = log_grp['latest_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
log_summary = log_grp[['cmdb_id', 'log_name', 'count', 'earliest', 'latest', 'p95_value']].sort_values(
    ['count', 'cmdb_id', 'log_name'], ascending=[False, True, True]
).head(50)

# Convert summaries to compact CSV-like strings for textual presentation
metric_csv = metric_summary.to_csv(index=False)
trace_csv = trace_summary.to_csv(index=False)
log_csv = log_summary.to_csv(index=False)

# Display the three CSV-like summary strings (compact)
metric_csv, trace_csv, log_csv

```
Out[1]:
```
summary = (
    "Summary of telemetry (global statistics computed over entire available series):\n\n"
    "1) metric.csv: Multiple (cmdb_id, kpi_name) series (most with 25 points) covering 2024-01-20T00:06:00Z → 2024-01-20T00:30:00Z. "
    "Notable global P95 values: carts-db diskio ≈ 2.5e6, orders mem ≈ 3.25e8, queue-master mem ≈ 2.80e8, front-end workload ≈ 24.8, carts socket = 12.0. "
    "CPU and latency P95s are moderate across services (examples: carts cpu ≈ 1.35, orders cpu ≈ 1.84, catalogue latency-90 ≈ 1.47).\n\n"
    "2) trace.csv: No trace series available (empty summary).\n\n"
    "3) log.csv: Multiple log.series (log.error_count and log.row_count). "
    "P95 of log.error_count is 0.0 for most services (no sustained errors detected globally). "
    "However log.row_count P95s are high for some components (front-end ≈ 2029, user ≈ 617, queue-master ≈ 583). "
    "A few log series have fewer samples (e.g., carts 22 points, carts-db 24, session-db 5).\n\n"
    "These summaries represent global thresholds (P95) computed from the full available time series and will be used for any further incident-window analysis."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

('cmdb_id,kpi_name,count,earliest,latest,p95_value\ncarts,cpu,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,1.3522726194665247\ncarts,latency-50,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,0.01993275345950956\ncarts,latency-90,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,0.04254131560761988\ncarts,mem,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,207894241.27999997\ncarts,socket,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,12.0\ncarts,workload,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,8.49169\ncarts-db,cpu,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,6.28575032878911\ncarts-db,diskio,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,2500967.949632468\ncarts-db,mem,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,96534801.06666666\ncarts-db,socket,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,7.0\ncatalogue,cpu,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,0.24066630349525112\ncatalogue,diskio,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,0.0\ncatalogue,latency-50,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,0.5902819079313231\ncatalogue,latency-90,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,1.4700947958978898\ncatalogue,mem,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,6725795.84\ncatalogue,socket,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,12.599999999999998\ncatalogue,workload,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,4.217116666666667\ncatalogue-db,cpu,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,0.2799370929219282\ncatalogue-db,diskio,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,0.0\ncatalogue-db,mem,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,432453290.6666667\ncatalogue-db,socket,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,4.743333333333332\nfront-end,cpu,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,4.83402294272681\nfront-end,error,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,0.0\nfront-end,latency-50,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,0.034949940512882575\nfront-end,latency-90,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,0.7428087693878873\nfront-end,mem,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,106108911.47228073\nfront-end,socket,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,13.52\nfront-end,workload,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,24.79881\norders,cpu,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,1.8389537321936111\norders,latency-50,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,0.03786397697023232\norders,latency-90,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,0.04815515854641834\norders,mem,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,324662679.4442105\norders,socket,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,18.136666666666663\norders,workload,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,2.1115766666666667\norders-db,cpu,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,0.43013975411045\norders-db,diskio,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,849372.5777051419\norders-db,mem,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,48282282.666666664\norders-db,socket,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,5.0\npayment,cpu,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,0.09455076265799366\npayment,latency-50,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,0.00256286295035144\npayment,latency-90,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,0.00451460054914746\npayment,mem,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,4613802.666666667\npayment,socket,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,6.0\npayment,workload,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,2.09597\nqueue-master,cpu,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,0.3772443981903566\nqueue-master,diskio,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,44791.85685216645\nqueue-master,mem,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,279937417.79087716\nqueue-master,socket,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,3.0\nrabbitmq,cpu,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,0.8405079965534947\nrabbitmq,diskio,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,0.0\n', 'cmdb_id,trace_name,count,earliest,latest,p95_value\n', 'cmdb_id,log_name,count,earliest,latest,p95_value\ncatalogue,log.error_count,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,0.0\ncatalogue,log.row_count,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,166.0\nfront-end,log.error_count,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,0.0\nfront-end,log.row_count,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,2028.8\norders,log.error_count,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,0.0\norders,log.row_count,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,129.6\npayment,log.error_count,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,0.0\npayment,log.row_count,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,104.8\nqueue-master,log.error_count,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,0.0\nqueue-master,log.row_count,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,583.1999999999999\nshipping,log.error_count,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,0.0\nshipping,log.row_count,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,64.8\nuser,log.error_count,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,0.0\nuser,log.row_count,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,616.8\ncarts-db,log.error_count,24,2024-01-20T00:07:00Z,2024-01-20T00:30:00Z,0.0\ncarts-db,log.row_count,24,2024-01-20T00:07:00Z,2024-01-20T00:30:00Z,1.0\ncarts,log.error_count,22,2024-01-20T00:06:00Z,2024-01-20T00:29:00Z,0.0\ncarts,log.row_count,22,2024-01-20T00:06:00Z,2024-01-20T00:29:00Z,100.0\nsession-db,log.error_count,5,2024-01-20T00:09:00Z,2024-01-20T00:29:00Z,0.0\nsession-db,log.row_count,5,2024-01-20T00:09:00Z,2024-01-20T00:29:00Z,5.0\n')```
```

In [ ]:
# In[2]:

import numpy as np
import pandas as pd

# Reuse already-loaded dataframes if present; otherwise load
try:
    metric
    log
except NameError:
    metric = pd.read_csv("metric.csv")
    log = pd.read_csv("log.csv")
    metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)
    log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)

# Ensure timestamps parsed (idempotent)
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)
log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)

# Incident window (UTC)
win_start = pd.to_datetime("2024-01-20 00:03:03", utc=True)
win_end   = pd.to_datetime("2024-01-20 00:33:03", utc=True)

# 1) Metric: compute global P95 per (cmdb_id,kpi_name) BEFORE filtering
metric_p95 = metric.groupby(['cmdb_id', 'kpi_name'], as_index=False)['value'].quantile(0.95)
metric_p95 = metric_p95.rename(columns={'value': 'p95_threshold'})

# Merge thresholds into metric and filter to window
metric_with_p95 = metric.merge(metric_p95, on=['cmdb_id', 'kpi_name'], how='left')
metric_win = metric_with_p95[(metric_with_p95['timestamp'] >= win_start) & (metric_with_p95['timestamp'] <= win_end)].copy()

# Identify anomalies: value >= p95_threshold
metric_win['is_anom'] = metric_win['value'] >= metric_win['p95_threshold']

# Aggregate per series where anomalies exist
anom_groups = (
    metric_win[metric_win['is_anom']]
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        p95_threshold=('p95_threshold', 'first'),
        anomaly_count_in_window=('value', 'count'),
        earliest_anomaly_timestamp=('timestamp', 'min'),
        latest_anomaly_timestamp=('timestamp', 'max'),
        max_value_in_window=('value', 'max')
    )
)

# Compute severity ratio, handling zero threshold
anom_groups['severity_ratio'] = np.where(
    anom_groups['p95_threshold'] == 0,
    np.inf,
    anom_groups['max_value_in_window'] / anom_groups['p95_threshold']
)

# Format timestamps as compact ISO UTC strings
if not anom_groups.empty:
    anom_groups['earliest_anomaly_timestamp'] = anom_groups['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    anom_groups['latest_anomaly_timestamp'] = anom_groups['latest_anomaly_timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

# Sort and limit to top 20 (by anomaly_count desc, then severity desc)
anom_summary = anom_groups.sort_values(
    by=['anomaly_count_in_window', 'severity_ratio'],
    ascending=[False, False]
).head(20)

# Reorder columns as requested
anom_summary = anom_summary[
    ['cmdb_id', 'kpi_name', 'p95_threshold', 'anomaly_count_in_window',
     'earliest_anomaly_timestamp', 'latest_anomaly_timestamp',
     'max_value_in_window', 'severity_ratio']
]

# 2) Traffic/business decrease detection: compute global P15 for workload and log.row_count BEFORE filtering

# Metric 'workload' P15 per cmdb_id
workload_full = metric[metric['kpi_name'] == 'workload']
workload_p15 = workload_full.groupby('cmdb_id', as_index=False)['value'].quantile(0.15).rename(columns={'value': 'p15_threshold'})

# Log 'log.row_count' P15 per cmdb_id
log_row_full = log[log['log_name'] == 'log.row_count']
logrow_p15 = log_row_full.groupby('cmdb_id', as_index=False)['value'].quantile(0.15).rename(columns={'value': 'p15_threshold'})

# Filter to window and find points <= p15

# Workload low points in window
workload_win = metric[
    (metric['kpi_name'] == 'workload') &
    (metric['timestamp'] >= win_start) & (metric['timestamp'] <= win_end)
].merge(workload_p15, on='cmdb_id', how='left')

workload_win['is_low'] = workload_win['value'] <= workload_win['p15_threshold']

work_low_groups = (
    workload_win[workload_win['is_low']]
    .groupby(['cmdb_id'], as_index=False)
    .agg(
        p15_threshold=('p15_threshold', 'first'),
        low_count_in_window=('value', 'count'),
        earliest_low_timestamp=('timestamp', 'min'),
        latest_low_timestamp=('timestamp', 'max'),
        min_value_in_window=('value', 'min')
    )
)
if not work_low_groups.empty:
    work_low_groups['kpi_name'] = 'workload'
    work_low_groups['severity_ratio'] = np.where(
        work_low_groups['p15_threshold'] == 0,
        np.inf,
        work_low_groups['min_value_in_window'] / work_low_groups['p15_threshold']
    )
    work_low_groups['earliest_low_timestamp'] = work_low_groups['earliest_low_timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    work_low_groups['latest_low_timestamp'] = work_low_groups['latest_low_timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

# log.row_count low points in window
logrow_win = log[
    (log['log_name'] == 'log.row_count') &
    (log['timestamp'] >= win_start) & (log['timestamp'] <= win_end)
].merge(logrow_p15, on='cmdb_id', how='left')

logrow_win['is_low'] = logrow_win['value'] <= logrow_win['p15_threshold']

log_low_groups = (
    logrow_win[logrow_win['is_low']]
    .groupby(['cmdb_id'], as_index=False)
    .agg(
        p15_threshold=('p15_threshold', 'first'),
        low_count_in_window=('value', 'count'),
        earliest_low_timestamp=('timestamp', 'min'),
        latest_low_timestamp=('timestamp', 'max'),
        min_value_in_window=('value', 'min')
    )
)
if not log_low_groups.empty:
    log_low_groups['kpi_name'] = 'log.row_count'
    log_low_groups['severity_ratio'] = np.where(
        log_low_groups['p15_threshold'] == 0,
        np.inf,
        log_low_groups['min_value_in_window'] / log_low_groups['p15_threshold']
    )
    log_low_groups['earliest_low_timestamp'] = log_low_groups['earliest_low_timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    log_low_groups['latest_low_timestamp'] = log_low_groups['latest_low_timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

# Combine low-detection results
low_summary = pd.concat(
    [df for df in (work_low_groups, log_low_groups) if (isinstance(df, pd.DataFrame) and not df.empty)],
    ignore_index=True,
    sort=False
)

# Reorder columns as requested
if not low_summary.empty:
    low_summary = low_summary[[
        'cmdb_id', 'kpi_name', 'p15_threshold', 'low_count_in_window',
        'earliest_low_timestamp', 'latest_low_timestamp',
        'min_value_in_window', 'severity_ratio'
    ]]
    # Sort by low_count desc then severity asc (smaller ratio = more severe)
    low_summary = low_summary.sort_values(by=['low_count_in_window', 'severity_ratio'], ascending=[False, True]).head(20)

# Convert summaries to compact CSV-like strings
anom_csv = anom_summary.to_csv(index=False) if not anom_summary.empty else "cmdb_id,kpi_name,p95_threshold,anomaly_count_in_window,earliest_anomaly_timestamp,latest_anomaly_timestamp,max_value_in_window,severity_ratio\n"
low_csv = low_summary.to_csv(index=False) if not low_summary.empty else "cmdb_id,kpi_name,p15_threshold,low_count_in_window,earliest_low_timestamp,latest_low_timestamp,min_value_in_window,severity_ratio\n"

# Display the two CSV-like summary strings
anom_csv, low_csv

```
Out[2]:
```
summary = (
    "Results (incident window 2024-01-20 00:03:03 → 2024-01-20 00:33:03 UTC):\n\n"
    "1) Series with values >= global P95 (top findings):\n"
    "- catalogue / diskio: global P95 = 0.0 but max in window = 86459.00 → severity = inf (large unexpected disk I/O spike)\n"
    "- catalogue-db / diskio and front-end / error and rabbitmq / diskio: P95 = 0.0 but values = 0.0 → severity = inf (P95 zero causes inf; these are borderline cases)\n"
    "- catalogue / cpu: P95 ≈ 0.2407, 2 anomaly points, max = 8.1186 → severity ≈ 33.73 (large CPU spikes)\n"
    "- orders / cpu: P95 ≈ 1.8390, 2 anomaly points, max = 6.7292 → severity ≈ 3.66\n"
    "- session-db / diskio: P95 ≈ 14363.90, 2 anomaly points, max = 28768.76 → severity ≈ 2.00\n"
    "- several socket metrics (carts-db, orders-db, payment, queue-master, rabbitmq, etc.) hit or slightly exceed their P95 (severity ≈ 1.0–1.16), e.g., carts.socket: p95=12.0, max≈12.28, severity≈1.02\n\n"
    "Interpretation: the most notable abnormality is the catalogue disk I/O spike (P95 was zero historically). CPU spikes on catalogue and orders and elevated DB disk I/O for session-db are also important.\n\n"
    "2) Traffic / business decrease detection (points <= global P15):\n"
    "- Severe decreases in log.row_count (strong indicators of reduced traffic or processing):\n"
    "  • user / log.row_count: p15 = 551.0, min = 25.0, count=5 → ratio ≈ 0.045 (very large drop)\n"
    "  • payment / log.row_count: p15 = 95.0, min = 5.0, count=5 → ratio ≈ 0.053\n"
    "  • orders / log.row_count: p15 = 110.0, min = 6.0, count=5 → ratio ≈ 0.055\n"
    "  • queue-master / log.row_count: p15 = 495.0, min = 27.0, count=5 → ratio ≈ 0.055\n"
    "  • shipping, catalogue, front-end, carts and others also show low log.row_count with ratios ~0.05–0.07 (notable traffic drop)\n"
    "- carts-db / log.row_count: p15 = 1.0, low_count = 23, min = 1.0 → ratio = 1.0 (many points equal the very low baseline)\n"
    "- workload KPIs show only modest decreases (payment, shipping, orders, catalogue, front-end, carts, user all have min/p15 ratios ≈ 0.95–0.98) — not severe.\n\n"
    "Overall conclusion:\n"
    "- There is a clear, system-wide reduction in business/traffic signals (log.row_count) across many services (user, payment, orders, queue-master, front-end, etc.) indicated by very low values relative to their global P15.\n"
    "- Concurrently, there is at least one striking infrastructure anomaly: a large unexpected disk I/O on the catalogue service (catalogue diskio) and CPU spikes on catalogue and orders. These could be related to the traffic change or a separate operational issue.\n\n"
    "Recommended next steps: focus on catalogue disk I/O and catalogue/orders CPU spikes for root-cause, and investigate why log.row_count (ingress/processing) dropped sharply for multiple services.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

('cmdb_id,kpi_name,p95_threshold,anomaly_count_in_window,earliest_anomaly_timestamp,latest_anomaly_timestamp,max_value_in_window,severity_ratio\ncatalogue,diskio,0.0,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,86459.00184049112,inf\ncatalogue-db,diskio,0.0,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,0.0,inf\nfront-end,error,0.0,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,0.0,inf\nrabbitmq,diskio,0.0,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,0.0,inf\ncarts-db,socket,7.0,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,7.0,1.0\norders-db,socket,5.0,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,5.0,1.0\npayment,socket,6.0,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,6.0,1.0\nqueue-master,socket,3.0,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,3.0,1.0\nrabbitmq,socket,11.0,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,11.0,1.0\nrabbitmq-exporter,mem,4669440.0,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,4669440.0,1.0\nrabbitmq-exporter,socket,1.0,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,1.0,1.0\nsession-db,socket,3.0,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,3.0,1.0\nuser,socket,17.0,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,17.0,1.0\nuser-db,socket,8.0,25,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,8.0,1.0\ncarts,socket,12.0,4,2024-01-20T00:24:00Z,2024-01-20T00:30:00Z,12.283333333333331,1.023611111111111\ncatalogue,cpu,0.24066630349525112,2,2024-01-20T00:06:00Z,2024-01-20T00:29:00Z,8.118586221660347,33.73378866817782\norders,cpu,1.8389537321936111,2,2024-01-20T00:14:00Z,2024-01-20T00:16:00Z,6.729195691217007,3.6592523092954767\nsession-db,diskio,14363.89538193676,2,2024-01-20T00:14:00Z,2024-01-20T00:30:00Z,28768.75892585396,2.00285216237595\nrabbitmq-exporter,cpu,0.0001613008050797799,2,2024-01-20T00:12:00Z,2024-01-20T00:21:00Z,0.0002175170041447,1.348517783510847\nshipping,socket,6.026666666666666,2,2024-01-20T00:10:00Z,2024-01-20T00:11:00Z,6.966666666666667,1.1559734513274338\n', 'cmdb_id,kpi_name,p15_threshold,low_count_in_window,earliest_low_timestamp,latest_low_timestamp,min_value_in_window,severity_ratio\ncarts-db,log.row_count,1.0,23,2024-01-20T00:07:00Z,2024-01-20T00:30:00Z,1.0,1.0\nuser,log.row_count,551.0,5,2024-01-20T00:14:00Z,2024-01-20T00:30:00Z,25.0,0.045372050816696916\npayment,log.row_count,95.0,5,2024-01-20T00:08:00Z,2024-01-20T00:30:00Z,5.0,0.05263157894736842\norders,log.row_count,110.0,5,2024-01-20T00:08:00Z,2024-01-20T00:30:00Z,6.0,0.05454545454545454\nqueue-master,log.row_count,495.0,5,2024-01-20T00:08:00Z,2024-01-20T00:30:00Z,27.0,0.05454545454545454\nshipping,log.row_count,55.0,5,2024-01-20T00:08:00Z,2024-01-20T00:30:00Z,3.0,0.05454545454545454\ncatalogue,log.row_count,153.0,5,2024-01-20T00:06:00Z,2024-01-20T00:30:00Z,10.0,0.06535947712418301\npayment,workload,1.8934,5,2024-01-20T00:15:00Z,2024-01-20T00:26:00Z,1.8133,0.9576951515791697\nsession-db,log.row_count,5.0,5,2024-01-20T00:09:00Z,2024-01-20T00:29:00Z,5.0,1.0\nfront-end,log.row_count,1830.2,4,2024-01-20T00:23:00Z,2024-01-20T00:30:00Z,100.0,0.054638837285542564\ncarts,log.row_count,28.749999999999996,4,2024-01-20T00:09:00Z,2024-01-20T00:29:00Z,25.0,0.8695652173913044\nshipping,workload,1.89725,4,2024-01-20T00:08:00Z,2024-01-20T00:25:00Z,1.8045,0.9511134536829621\norders,workload,1.90133,4,2024-01-20T00:18:00Z,2024-01-20T00:26:00Z,1.81795,0.9561464869328312\ncatalogue,workload,3.8129299999999997,4,2024-01-20T00:08:00Z,2024-01-20T00:25:00Z,3.6823166666666665,0.9657446285839674\nfront-end,workload,23.36807333333333,4,2024-01-20T00:16:00Z,2024-01-20T00:25:00Z,22.6822,0.9706491278271124\ncarts,workload,7.54278,4,2024-01-20T00:16:00Z,2024-01-20T00:26:00Z,7.335733333333334,0.9725503505780805\nuser,workload,17.01896,4,2024-01-20T00:16:00Z,2024-01-20T00:26:00Z,16.613266666666664,0.9761622723519336\n')```
```

In [ ]:
# In[3]:

import pandas as pd
import os

# Reuse existing variables if present; otherwise load metric/log already loaded earlier
try:
    error_logs
except NameError:
    error_logs = pd.read_csv("error_logs.csv")

# Parse timestamps as UTC datetimes (Unix seconds)
error_logs['timestamp'] = pd.to_datetime(error_logs['timestamp'], unit='s', utc=True)

# Incident window (UTC)
win_start = pd.to_datetime("2024-01-20 00:03:03", utc=True)
win_end   = pd.to_datetime("2024-01-20 00:33:03", utc=True)

# Filter to incident window (inclusive)
err_win = error_logs[(error_logs['timestamp'] >= win_start) & (error_logs['timestamp'] <= win_end)].copy()

# Normalize message: replace empty/NaN with 'NA' and collapse whitespace
err_win['message'] = err_win['message'].fillna('').astype(str).map(lambda s: s.strip() if s.strip() else 'NA')

# Group by cmdb_id and aggregate
def sample_messages(s, n=5):
    # take up to n sample messages from the group's rows in time order
    msgs = s.head(n).tolist()
    if not msgs:
        return 'NA'
    return " | ".join(msgs)

grouped = (
    err_win.sort_values('timestamp')
    .groupby('cmdb_id', as_index=False)
    .agg(
        count_of_entries_in_window=('message', 'count'),
        earliest_timestamp=('timestamp', 'min'),
        latest_timestamp=('timestamp', 'max'),
        sample_messages_raw=('message', lambda s: sample_messages(s, n=5))
    )
)

# Format timestamps as compact ISO UTC strings
if not grouped.empty:
    grouped['earliest_timestamp'] = grouped['earliest_timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    grouped['latest_timestamp'] = grouped['latest_timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

# Reorder and sort by count desc, limit top 20
err_summary = grouped[['cmdb_id', 'count_of_entries_in_window', 'earliest_timestamp', 'latest_timestamp', 'sample_messages_raw']].sort_values(
    by='count_of_entries_in_window', ascending=False
).head(20)

# Rename sample column to 'sample_messages' for output clarity
err_summary = err_summary.rename(columns={'sample_messages_raw': 'sample_messages'})

# Convert to CSV-like string for compact textual output
error_csv = err_summary.to_csv(index=False)

# Display the CSV-like result
error_csv

```
Out[3]:
```
summary = (
    "I scanned error_logs.csv for the incident window 2024-01-20 00:03:03 → 2024-01-20 00:33:03 UTC and found no entries. "
    "The query returned only the CSV header and no rows, so there are no error_log records to report per cmdb_id within that window."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id,count_of_entries_in_window,earliest_timestamp,latest_timestamp,sample_messages```
```